# 01. ML pipeline: train/test, CV, baseline i leakage

Ten notebook jest **brakującym fundamentem** przed blokiem liniowym i logistycznym.
Nie chodzi jeszcze o konkretny algorytm, tylko o sposób myślenia:

$$
\text{dane} \rightarrow \text{podział} \rightarrow \text{fit na train} \rightarrow \text{ocena na test}
$$

Po tej lekcji student powinien umieć odpowiedzieć na pytania:

1. Dlaczego nie oceniamy modelu na tych samych danych, na których go uczymy?
2. Co to jest baseline?
3. Co to jest cross-validation?
4. Dlaczego leakage potrafi dać „magicznie dobry” wynik?

## 1. Minimalny schemat pipeline'u

W każdym zadaniu ML trzeba jasno oddzielić trzy rzeczy:

- **cechy** $X$ — informacje dostępne przed predykcją,
- **target** $y$ — to, co chcemy przewidzieć,
- **metrykę** — sposób oceny jakości modelu.

Najprostszy pipeline:

$$
(X, y) \rightarrow (X_{train}, X_{test}, y_{train}, y_{test})
$$

$$
\text{model.fit}(X_{train}, y_{train})
$$

$$
\text{model.predict}(X_{test})
$$

Dydaktyczna intuicja: **test set ma udawać przyszłość**. Model nie powinien widzieć przyszłości podczas nauki.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score

pd.set_option('display.precision', 4)
rng = np.random.default_rng(42)

## 2. Mały zbiór danych: nauka i wynik

Zrobimy syntetyczny zbiór przypominający prosty case edukacyjny.

Cechy:

- `StudyHours` — liczba godzin nauki,
- `Attendance` — obecność na zajęciach,
- `PreviousScore` — wcześniejszy wynik.

Targety:

- `Grade` — wynik punktowy, regresja,
- `Pass` — czy student zdał, klasyfikacja.

In [2]:
n = 90
study_hours = rng.uniform(0, 12, n)
attendance = rng.uniform(40, 100, n)
previous_score = rng.uniform(20, 95, n)
noise = rng.normal(0, 8, n)

grade = 25 + 4.2 * study_hours + 0.18 * attendance + 0.28 * previous_score + noise
grade = np.clip(grade, 0, 100)
pass_exam = (grade >= 60).astype(int)

df = pd.DataFrame({
    'StudyHours': study_hours,
    'Attendance': attendance,
    'PreviousScore': previous_score,
    'Grade': grade,
    'Pass': pass_exam
})

df.head()

,StudyHours,Attendance,PreviousScore,Grade,Pass
0,9.2875,49.1387,86.8094,95.0279,1
1,5.2665,81.7792,87.0085,88.0595,1
2,10.3032,66.7694,58.9144,92.3452,1
3,8.3684,62.8613,43.6947,87.4692,1
4,1.1301,58.0907,77.9009,70.1169,1


In [3]:
print(df[['Grade', 'Pass']].describe())
print('\nLiczba zdanych / niezdanych:')
print(df['Pass'].value_counts().rename(index={0:'Fail', 1:'Pass'}))

          Grade     Pass
count   90.0000  90.0000
mean    77.8226   0.8111
std     16.8048   0.3936
min     38.7956   0.0000
25%     68.6136   1.0000
50%     80.3506   1.0000
75%     91.1898   1.0000
max    100.0000   1.0000

Liczba zdanych / niezdanych:
Pass
Pass    73
Fail    17
Name: count, dtype: int64


## 3. Podział train/test

Najprostszy podział:

$$
\text{train} = 75\%, \quad \text{test} = 25\%
$$

Uwaga dydaktyczna: podział robimy **przed** treningiem i przed oceną. Jeżeli skalujemy, imputujemy braki albo wybieramy cechy, to te operacje też powinny być uczone tylko na train.

In [4]:
features = ['StudyHours', 'Attendance', 'PreviousScore']
X = df[features]
y_reg = df['Grade']
y_cls = df['Pass']

X_train, X_test, y_train_reg, y_test_reg, y_train_cls, y_test_cls = train_test_split(
    X, y_reg, y_cls, test_size=0.25, random_state=7, stratify=y_cls
)

print('Train:', X_train.shape, 'Test:', X_test.shape)

Train: (67, 3) Test: (23, 3)


## 4. Baseline w regresji

Baseline to prosty punkt odniesienia. Dla regresji naturalny baseline to przewidywanie średniej z treningu:

$$
\hat y_i = \bar y_{train}
$$

Model ma sens dopiero wtedy, gdy bije taki naiwny punkt odniesienia.

In [5]:
# Baseline: zawsze przewiduj średnią z y_train
baseline_pred = np.repeat(y_train_reg.mean(), len(y_test_reg))

# Model liniowy
lin = LinearRegression()
lin.fit(X_train, y_train_reg)
lin_pred = lin.predict(X_test)

results_reg = pd.DataFrame({
    'model': ['baseline_mean', 'linear_regression'],
    'MAE': [mean_absolute_error(y_test_reg, baseline_pred), mean_absolute_error(y_test_reg, lin_pred)],
    'RMSE': [mean_squared_error(y_test_reg, baseline_pred) ** 0.5, mean_squared_error(y_test_reg, lin_pred) ** 0.5],
    'R2': [r2_score(y_test_reg, baseline_pred), r2_score(y_test_reg, lin_pred)]
})
results_reg

,model,MAE,RMSE,R2
0,baseline_mean,14.8367,17.7563,-0.0066
1,linear_regression,6.8644,8.4217,0.7736


Interpretacja:

- `MAE` mówi: średnio o ile punktów się mylimy.
- `RMSE` mocniej karze duże błędy.
- $R^2$ mówi, ile zmienności targetu wyjaśnia model względem baseline'u średniej:

$$
R^2 = 1 - \frac{SSE_{model}}{SSE_{baseline}}
$$

## 5. Baseline w klasyfikacji

Dla klasyfikacji prosty baseline to przewidywanie najczęstszej klasy.

Jeśli większość studentów zdała, baseline mówi zawsze: „zdał”.

In [6]:
majority_class = int(y_train_cls.mode()[0])
baseline_cls_pred = np.repeat(majority_class, len(y_test_cls))

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train_cls)
proba = logreg.predict_proba(X_test)[:, 1]
pred_cls = (proba >= 0.5).astype(int)

results_cls = pd.DataFrame({
    'model': ['baseline_majority', 'logistic_regression'],
    'accuracy': [accuracy_score(y_test_cls, baseline_cls_pred), accuracy_score(y_test_cls, pred_cls)],
    'AUC': [np.nan, roc_auc_score(y_test_cls, proba)]
})
results_cls

,model,accuracy,AUC
0,baseline_majority,0.8261,NaN
1,logistic_regression,0.8696,0.9079


In [7]:
cm = confusion_matrix(y_test_cls, pred_cls)
pd.DataFrame(cm, index=['actual_0_fail', 'actual_1_pass'], columns=['pred_0_fail', 'pred_1_pass'])

,pred_0_fail,pred_1_pass
actual_0_fail,2,2
actual_1_pass,1,18


## 6. Cross-validation

Jeden podział train/test może być pechowy. Cross-validation powtarza eksperyment na kilku podziałach.

Dla $K=5$:

$$
\text{wynik CV} = \frac{1}{5}\sum_{k=1}^{5} score_k
$$

Intuicja: każdy fragment danych raz udaje test, a kilka razy udaje train.

In [8]:
# Tworzymy obiekt KFold do walidacji krzyżowej (cross-validation)
kfold = KFold(
    n_splits=5,        # dzielimy dane na 5 części (foldów)
    shuffle=True,      # mieszamy dane przed podziałem
    random_state=123   # ustawiamy seed losowości -> wynik będzie powtarzalny
)

# cross_val_score:
# 1. bierze model (LinearRegression)
# 2. wielokrotnie trenuje go na części danych
# 3. testuje na pozostałej części
# 4. zwraca wynik dla każdego foldu
cv_scores = cross_val_score(
    LinearRegression(), # model regresji liniowej
    X,                   # cechy wejściowe
    y_reg,               # wartości docelowe
    cv=kfold,            # sposób podziału danych (nasz KFold)
    scoring='r2'         # metryka jakości -> współczynnik R²
)

# Wyniki R² dla każdego z 5 foldów
print('R2 w foldach:', np.round(cv_scores, 3))

# Średni wynik modelu
print('Średnie R2:', round(cv_scores.mean(), 3))

# Odchylenie standardowe wyników
# pokazuje stabilność modelu między foldami
print('Odchylenie:', round(cv_scores.std(), 3))

R2 w foldach: [0.822 0.688 0.701 0.873 0.745]
Średnie R2: 0.766
Odchylenie: 0.071


## Współczynnik determinacji $R^2$

Współczynnik $R^2$ mówi nam, jak dobrze model wyjaśnia zmienność danych.

Porównujemy:

- błąd modelu,
- z błędem bardzo prostego modelu bazowego przewidującego tylko średnią.

Definicja:

$$
R^2 = 1 - \frac{SSE}{TSS}
$$

gdzie:

$$
SSE = \sum_i (y_i - \hat{y}_i)^2
$$

to suma kwadratów błędów modelu,

natomiast:

$$
TSS = \sum_i (y_i - \bar{y})^2
$$

to całkowita zmienność danych względem średniej.

---

Interpretacja:

- $R^2 = 1$
  
  model idealny,
  wszystkie punkty leżą dokładnie na przewidywaniach.

- $R^2 = 0$
  
  model nie jest lepszy niż zgadywanie średniej.

- $R^2 < 0$
  
  model działa gorzej niż przewidywanie samej średniej.

---

Intuicyjnie:

$$
R^2
$$

mierzy, jaką część zmienności danych udało się wyjaśnić modelowi.

Przykład:

$$
R^2 = 0.85
$$

oznacza, że model wyjaśnia około $85\%$ zmienności danych.


- $\bar{y}$ to średnia wszystkich wartości w danych:

$$
\bar{y} = \frac{1}{n}\sum_i y_i
$$

- $\hat{y}_i$ to przewidywanie modelu dla konkretnego punktu $x_i$:

$$
\hat{y}_i = b_0 + b_1 x_i
$$

- $\bar{y}$ odpowiada modelowi „zgaduję zawsze średnią”.  
- $\hat{y}_i$ wykorzystuje zależność między $x$ i $y$.  

- $\bar{y}$ to średnia wszystkich wartości w danych — jedna stała liczba.  
- $\hat{y}_i$ to przewidywanie modelu dla konkretnego punktu $x_i$.  
- $\bar{y}$ odpowiada modelowi „zgaduję zawsze średnią”.  
- $\hat{y}_i$ wykorzystuje zależność między $x$ i $y$.  

## 7. Leakage: kiedy model zna odpowiedź z przyszłości

**Leakage** występuje wtedy, gdy do cech trafi informacja, której nie mielibyśmy w momencie predykcji.

Przykład skrajny: chcemy przewidzieć `Pass`, ale dodajemy cechę prawie równą `Grade`.

Formalnie model widzi wtedy coś bardzo bliskiego targetowi:

$$
X_{leak} \approx y
$$

Wynik testowy wygląda świetnie, ale taki model jest bezużyteczny w realnym użyciu.

In [9]:
df_leak = df.copy()
df_leak['GradeAfterExam_LEAK'] = df_leak['Grade'] + rng.normal(0, 1, len(df_leak))

X_honest = df_leak[['StudyHours', 'Attendance', 'PreviousScore']]
X_leaky = df_leak[['StudyHours', 'Attendance', 'PreviousScore', 'GradeAfterExam_LEAK']]
y = df_leak['Pass']

Xh_train, Xh_test, Xl_train, Xl_test, yh_train, yh_test = train_test_split(
    X_honest, X_leaky, y, test_size=0.25, random_state=11, stratify=y
)

model_honest = LogisticRegression(max_iter=1000).fit(Xh_train, yh_train)
model_leaky = LogisticRegression(max_iter=1000).fit(Xl_train, yh_train)

auc_honest = roc_auc_score(yh_test, model_honest.predict_proba(Xh_test)[:, 1])
auc_leaky = roc_auc_score(yh_test, model_leaky.predict_proba(Xl_test)[:, 1])

pd.DataFrame({
    'model': ['uczciwy_model', 'model_z_leakage'],
    'AUC_test': [auc_honest, auc_leaky]
})

,model,AUC_test
0,uczciwy_model,0.8421
1,model_z_leakage,0.9605


### Komentarz do prowadzenia

Wysokie AUC modelu z leakage nie oznacza, że model jest mądry. Oznacza, że w danych pojawiła się informacja, której nie wolno użyć.

Pytanie kontrolne dla studentów:

> Czy ta cecha byłaby znana w chwili, gdy model ma podjąć decyzję?

Jeśli odpowiedź brzmi „nie”, prawdopodobnie mamy leakage.

## 8. Checklista pipeline'u

Przed każdym modelem sprawdź:

1. Co jest targetem $y$?
2. Jakie cechy są dostępne przed predykcją?
3. Jaki jest baseline?
4. Jaki mamy podział train/test albo CV?
5. Jaką metrykę optymalizujemy?
6. Czy żadna cecha nie przecieka z targetu?
7. Czy wynik jest stabilny na kilku podziałach?

## Proste ćwiczenia

1. Wskaż baseline dla regresji i klasyfikacji w tym notebooku.
2. Dlaczego model z `GradeAfterExam_LEAK` ma bardzo dobre AUC?
3. Co może pójść źle, jeśli wybierzemy cechy na całym zbiorze przed podziałem train/test?
4. Zmień `test_size` z `0.25` na `0.4`. Czy wyniki są identyczne?

## 9. Mini-lab: Titanic — poprawny pipeline i leakage

Ten dodatek spina tematy z notebooka na jednym prawdziwym zbiorze danych.

Przewidujemy:

$$
y = \text{survived}
$$

Pytanie przewodnie:

$$
\text{Czy wysoki wynik modelu oznacza dobry model, czy błąd w pipeline?}
$$

Porównamy poprawny pipeline z celowym błędem typu **data leakage**.


In [10]:
from pathlib import Path

def find_data_file(filename):
    """Znajduje plik CSV niezależnie od tego, z jakiego katalogu uruchamiamy notebook."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd().parent / filename,
        Path.cwd() / 'data' / filename,
        Path.cwd().parent / 'data' / filename,
        Path('/mnt/data') / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Nie znaleziono pliku: {filename}')

def make_onehot_encoder():
    """Kompatybilność ze starszym i nowszym scikit-learn: sparse vs sparse_output."""
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

titanic = pd.read_csv(find_data_file('seaborn_titanic.csv'))
print('Rozmiar danych:', titanic.shape)
titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'sibsp', 'parch', 'embarked', 'alive']].head()


Rozmiar danych: (891, 15)


,survived,pclass,sex,age,fare,sibsp,parch,embarked,alive
0,0,3,male,22.0,7.2500,1,0,S,no
1,1,1,female,38.0,71.2833,1,0,C,yes
2,1,3,female,26.0,7.9250,0,0,S,yes
3,1,1,female,35.0,53.1000,1,0,S,yes
4,0,3,male,35.0,8.0500,0,0,S,no


### 9.1. Cechy, target i podział danych

Używamy cech, które mogłyby być dostępne przed predykcją:

$$
X = \{age, fare, sibsp, parch, sex, pclass, embarked\}
$$

Nie używamy `alive`, bo to jest praktycznie tekstowa wersja odpowiedzi.


In [11]:
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['sex', 'pclass', 'embarked']

X = titanic[numeric_features + categorical_features]
y = titanic['survived'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print('Train:', X_train.shape)
print('Test :', X_test.shape)
print('\nProporcja klas w train:')
print(y_train.value_counts(normalize=True).rename('proportion'))


Train: (623, 7)
Test : (268, 7)

Proporcja klas w train:
survived
0    0.6164
1    0.3836
Name: proportion, dtype: float64


### 9.2. Baseline i poprawny pipeline

Baseline odpowiada na pytanie:

$$
\text{Czy model jest lepszy niż bardzo prosta reguła?}
$$

Pipeline pilnuje, że imputacja, skalowanie i one-hot encoding są uczone tylko na `train`.


In [12]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', make_onehot_encoder())
])

preprocess = ColumnTransformer([
    ('num', num_pipe, numeric_features),
    ('cat', cat_pipe, categorical_features)
])

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

clean_pipeline = Pipeline([
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000))
])
clean_pipeline.fit(X_train, y_train)
clean_pred = clean_pipeline.predict(X_test)
clean_score = clean_pipeline.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    {
        'model': 'baseline: najczęstsza klasa',
        'accuracy': accuracy_score(y_test, baseline_pred),
        'f1': f1_score(y_test, baseline_pred, zero_division=0),
        'auc': np.nan,
    },
    {
        'model': 'LogisticRegression bez leakage',
        'accuracy': accuracy_score(y_test, clean_pred),
        'f1': f1_score(y_test, clean_pred, zero_division=0),
        'auc': roc_auc_score(y_test, clean_score),
    }
])
results


,model,accuracy,f1,auc
0,baseline: najczęstsza klasa,0.6157,0.0000,NaN
1,LogisticRegression bez leakage,0.7985,0.7273,0.8485


### 9.3. Celowy błąd: leakage przez `alive`

Kolumna `alive` mówi prawie dokładnie to samo co `survived`.

Jeśli dodamy ją do $X$, model dostaje odpowiedź ukrytą w cechach:

$$
X_{leak} = X \cup \{alive\}
$$

To nie jest lepszy model. To błąd w danych.


In [13]:
leak_categorical_features = categorical_features + ['alive']
X_leak = titanic[numeric_features + leak_categorical_features]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y, test_size=0.30, random_state=42, stratify=y
)

preprocess_leak = ColumnTransformer([
    ('num', num_pipe, numeric_features),
    ('cat', cat_pipe, leak_categorical_features)
])

leak_pipeline = Pipeline([
    ('preprocess', preprocess_leak),
    ('model', LogisticRegression(max_iter=2000))
])
leak_pipeline.fit(X_train_l, y_train_l)
leak_pred = leak_pipeline.predict(X_test_l)
leak_score = leak_pipeline.predict_proba(X_test_l)[:, 1]

leak_row = pd.DataFrame([{
    'model': 'LogisticRegression z leakage: dodano alive',
    'accuracy': accuracy_score(y_test_l, leak_pred),
    'f1': f1_score(y_test_l, leak_pred, zero_division=0),
    'auc': roc_auc_score(y_test_l, leak_score),
}])

pd.concat([results, leak_row], ignore_index=True)


,model,accuracy,f1,auc
0,baseline: najczęstsza klasa,0.6157,0.0000,NaN
1,LogisticRegression bez leakage,0.7985,0.7273,0.8485
2,LogisticRegression z leakage: dodano alive,1.0000,1.0000,1.0000


### 9.4. Wniosek

Przy podejrzanie dobrym wyniku warto zapytać:

$$
\text{Czy model nie dostał kopii targetu albo informacji z przyszłości?}
$$

Poprawna kolejność to:

$$
\text{split} \rightarrow \text{fit preprocessingu na train} \rightarrow \text{fit modelu na train} \rightarrow \text{ocena na test}
$$
